# 17 — DBFS and DBUtils

**What you will learn:**
- What DBFS is and how it differs from actual cloud storage
- `dbutils.fs` — navigate, read, write, copy, move, delete files
- `dbutils.secrets` — retrieve secrets without hardcoding credentials
- `dbutils.widgets` — parameterize notebooks with runtime inputs
- `dbutils.notebook` — chain notebooks together, pass values between them
- `dbutils.jobs` — get context about the running job
- DBFS mounts (legacy) vs Unity Catalog Volumes (modern)
- Interview cheat-sheet: what to say about each

---
## Part 1 — What is DBFS?

**DBFS (Databricks File System)** is a distributed virtual file system that Databricks layers on top of cloud object storage (Azure Blob / ADLS Gen2 on Azure, S3 on AWS).  
It presents a Unix-like path API so you can use familiar `/path/to/file` syntax instead of `abfss://container@account.dfs.core.windows.net/path`.

```
What you write           What DBFS maps it to
─────────────────────    ────────────────────────────────────────────────
/FileStore/...     →     Databricks-managed blob (UI uploads, small files)
/mnt/datalake/...  →     Your ADLS Gen2 container (mount configured once)
/tmp/...           →     Ephemeral per-cluster scratch space
/Volumes/...       →     Unity Catalog Volume (modern, preferred)
```

### Key facts for interviews
| Fact | Detail |
|---|---|
| DBFS is not a real filesystem | It is an abstraction; actual bytes live in cloud storage |
| `/tmp/` is cluster-local | Files disappear when the cluster terminates |
| `/FileStore/` is persistent | Backed by managed Databricks storage; survives cluster restarts |
| Mounts are legacy | Unity Catalog External Locations + Volumes replace them |
| DBFS is disabled by default | On Unity Catalog clusters, DBFS root access is restricted |

---
## Part 2 — dbutils Overview

`dbutils` is a Databricks-provided utility object available in every notebook.  
It is **not** a Python package — it is injected by the Databricks runtime.

```
dbutils
├── fs          — file system operations (ls, cp, mv, rm, put, head ...)
├── secrets     — retrieve secrets from Secret Scopes
├── widgets     — notebook parameters (text, dropdown, multiselect, combobox)
├── notebook    — run child notebooks, pass/return values
├── jobs        — get job run context (job ID, run ID, task key)
└── library     — install libraries at runtime (rarely needed now)
```

**Get help inline:**

In [ ]:
# List all dbutils modules
dbutils.help()

In [ ]:
# Help for a specific module
dbutils.fs.help()

In [ ]:
# Help for a specific command
dbutils.fs.help("ls")

---
## Part 3 — dbutils.fs (File System Operations)

All `dbutils.fs` commands operate on DBFS paths.  
They also work on Unity Catalog Volume paths (`/Volumes/...`) and, if configured, `abfss://` paths.

### Command Reference

| Command | Purpose |
|---|---|
| `dbutils.fs.ls(path)` | List files/directories at a path |
| `dbutils.fs.put(path, content, overwrite)` | Write a string to a file |
| `dbutils.fs.head(path, maxBytes)` | Read the first N bytes of a file |
| `dbutils.fs.cp(src, dst, recurse)` | Copy a file or directory |
| `dbutils.fs.mv(src, dst, recurse)` | Move/rename a file or directory |
| `dbutils.fs.rm(path, recurse)` | Delete a file or directory |
| `dbutils.fs.mkdirs(path)` | Create a directory (and parents) |
| `dbutils.fs.mount(...)` | Mount cloud storage (legacy — avoid with Unity Catalog) |
| `dbutils.fs.mounts()` | List all current mounts |

In [ ]:
# Use FileStore so files persist across cluster restarts
# /FileStore is backed by Databricks-managed storage — safe for practice
PRACTICE_DIR = "/FileStore/practice_dbutils"

# Create the directory
dbutils.fs.mkdirs(PRACTICE_DIR)
print(f"Directory created: {PRACTICE_DIR}")

In [ ]:
# ── dbutils.fs.put ──────────────────────────────────────────────────────────
# Write a string directly to a file in DBFS
# overwrite=True replaces any existing file at that path

csv_content = """id,city,population
1,Seattle,737000
2,Portland,652000
3,Denver,715000
4,Austin,978000
"""

dbutils.fs.put(f"{PRACTICE_DIR}/cities.csv", csv_content, overwrite=True)
print("File written.")

In [ ]:
# ── dbutils.fs.ls ────────────────────────────────────────────────────────────
# Returns a list of FileInfo objects: name, path, size, modificationTime

files = dbutils.fs.ls(PRACTICE_DIR)
for f in files:
    print(f"name={f.name}  path={f.path}  size={f.size} bytes")

# display() renders it as a table — more readable in a notebook
display(dbutils.fs.ls(PRACTICE_DIR))

In [ ]:
# ── dbutils.fs.head ──────────────────────────────────────────────────────────
# Quick peek at file contents — returns first maxBytes as a string
# Useful for sanity-checking large files without loading them into Spark

preview = dbutils.fs.head(f"{PRACTICE_DIR}/cities.csv", maxBytes=256)
print(preview)

In [ ]:
# ── dbutils.fs.cp ────────────────────────────────────────────────────────────
# Copy a single file
dbutils.fs.cp(
    f"{PRACTICE_DIR}/cities.csv",
    f"{PRACTICE_DIR}/cities_backup.csv"
)

# Copy a whole directory (recurse=True)
dbutils.fs.mkdirs(f"{PRACTICE_DIR}/subdir")
dbutils.fs.put(f"{PRACTICE_DIR}/subdir/notes.txt", "hello world", overwrite=True)
dbutils.fs.cp(f"{PRACTICE_DIR}/subdir", f"{PRACTICE_DIR}/subdir_copy", recurse=True)

display(dbutils.fs.ls(PRACTICE_DIR))

In [ ]:
# ── dbutils.fs.mv ────────────────────────────────────────────────────────────
# Move/rename — same API as cp, but the source is deleted after move
dbutils.fs.mv(
    f"{PRACTICE_DIR}/cities_backup.csv",
    f"{PRACTICE_DIR}/cities_renamed.csv"
)
display(dbutils.fs.ls(PRACTICE_DIR))

In [ ]:
# ── dbutils.fs.rm ────────────────────────────────────────────────────────────
# Delete a single file
dbutils.fs.rm(f"{PRACTICE_DIR}/cities_renamed.csv")

# Delete a directory and all its contents (recurse=True is required for directories)
dbutils.fs.rm(f"{PRACTICE_DIR}/subdir_copy", recurse=True)

display(dbutils.fs.ls(PRACTICE_DIR))

In [ ]:
# ── Reading DBFS files with Spark ────────────────────────────────────────────
# Once files are in DBFS, Spark can read them using the same path

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{PRACTICE_DIR}/cities.csv")
)
df.show()

# Write back to DBFS
df.write.mode("overwrite").parquet(f"{PRACTICE_DIR}/cities_parquet/")
display(dbutils.fs.ls(f"{PRACTICE_DIR}/cities_parquet/"))

---
## Part 4 — DBFS Mounts (Legacy)

A **mount** maps a cloud storage container to a DBFS path so notebooks can use `/mnt/name/` instead of  
`abfss://container@account.dfs.core.windows.net/`.

> **Status:** Mounts are **legacy**. Unity Catalog External Locations and Volumes replace them.  
> On Unity Catalog clusters, DBFS mount creation may be disabled.  
> The patterns below are shown for **interview awareness** — you will encounter them in older codebases.

### How mounts worked (reference only)

```python
# Mount ADLS Gen2 using a service principal (legacy pattern)
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id":     "<app-id>",
    "fs.azure.account.oauth2.client.secret": dbutils.secrets.get("my-scope", "sp-secret"),
    "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/<tenant-id>/oauth2/token",
}

dbutils.fs.mount(
    source      = "abfss://bronze@mystorageaccount.dfs.core.windows.net/",
    mount_point = "/mnt/bronze",
    extra_configs = configs
)
```

After mounting, every notebook in the workspace can use `/mnt/bronze/`:
```python
df = spark.read.parquet("/mnt/bronze/sales/2024/")
```

In [ ]:
# List all current mounts in the workspace
# On Unity Catalog clusters this may show only the built-in /databricks-datasets mount
display(dbutils.fs.mounts())

In [ ]:
# Databricks ships sample datasets under /databricks-datasets — always available
display(dbutils.fs.ls("/databricks-datasets/"))

In [ ]:
# Read one of the built-in sample datasets (no setup needed)
df_diamonds = spark.read.csv(
    "/databricks-datasets/Rdatasets/data-001/csv/ggplot2/diamonds.csv",
    header=True, inferSchema=True
)
print(f"Rows: {df_diamonds.count()}")
df_diamonds.show(5)

---
## Part 5 — dbutils.secrets

**Secret Scopes** store credentials (storage keys, passwords, tokens) outside notebooks.  
Instead of hardcoding `password = "abc123"`, you call `dbutils.secrets.get(scope, key)`.

### Two types of Secret Scopes

| Type | Backed by | Setup |
|---|---|---|
| **Databricks-managed** | Databricks internal storage | Create via Databricks CLI or `#secrets/createScope` URL |
| **Azure Key Vault-backed** | Azure Key Vault | Link scope to Key Vault URI + resource ID |

### Setup (one time, done outside the notebook)
```bash
# Using Databricks CLI
databricks secrets create-scope my-scope
databricks secrets put-secret my-scope storage-account-key --string-value "<key>"
databricks secrets put-secret my-scope db-password         --string-value "<pwd>"
```

### Reading secrets in a notebook

In [ ]:
# ── dbutils.secrets.listScopes ───────────────────────────────────────────────
# List all secret scopes visible to your user
scopes = dbutils.secrets.listScopes()
for s in scopes:
    print(s.name)

In [ ]:
# ── dbutils.secrets.list ─────────────────────────────────────────────────────
# List keys in a scope (values are always redacted — you only see key names)
# Replace "my-scope" with your actual scope name

# keys = dbutils.secrets.list("my-scope")
# for k in keys:
#     print(k.key)

print("Uncomment and replace 'my-scope' with your scope name.")

In [ ]:
# ── dbutils.secrets.get ──────────────────────────────────────────────────────
# Retrieve a secret value — Databricks REDACTS it in notebook output (prints [REDACTED])
# You cannot print the actual value; it can only be passed to functions

# storage_key = dbutils.secrets.get(scope="my-scope", key="storage-account-key")
# spark.conf.set(
#     "fs.azure.account.key.mystorageaccount.dfs.core.windows.net",
#     storage_key          # safe — value never appears in output
# )

print("Secret values are always [REDACTED] in Databricks notebook output — this is intentional.")

---
## Part 6 — dbutils.widgets (Notebook Parameters)

Widgets add an interactive UI element at the top of the notebook — like a form field.  
They also allow external callers (ADF, Databricks Jobs, `dbutils.notebook.run`) to pass parameter values in.

### Widget types

| Type | Use when |
|---|---|
| `text` | Free-form string input |
| `dropdown` | Fixed set of choices (one selection) |
| `combobox` | Fixed suggestions but free-form input also allowed |
| `multiselect` | Fixed set of choices (multiple selections) |

In [ ]:
# ── Create widgets ────────────────────────────────────────────────────────────
# These appear as UI controls at the top of the notebook

# Text widget: name, default_value, label
dbutils.widgets.text("run_date", "2024-01-01", "Run Date")

# Dropdown widget: name, default_value, choices, label
dbutils.widgets.dropdown(
    "environment",
    "dev",
    ["dev", "staging", "prod"],
    "Environment"
)

# Combobox: like dropdown but user can also type a custom value
dbutils.widgets.combobox(
    "department",
    "Engineering",
    ["Engineering", "Marketing", "HR"],
    "Department"
)

# Multiselect: user can pick multiple values
dbutils.widgets.multiselect(
    "regions",
    "US",
    ["US", "EU", "APAC"],
    "Regions"
)

print("Widgets created — look for the controls above this cell.")

In [ ]:
# ── Read widget values ────────────────────────────────────────────────────────
# dbutils.widgets.get always returns a string — cast if needed

run_date    = dbutils.widgets.get("run_date")
environment = dbutils.widgets.get("environment")
department  = dbutils.widgets.get("department")
regions_str = dbutils.widgets.get("regions")   # comma-separated string for multiselect
regions     = regions_str.split(",")           # convert to list

print(f"run_date    : {run_date}")
print(f"environment : {environment}")
print(f"department  : {department}")
print(f"regions     : {regions}")

In [ ]:
# ── Use widget values in SQL cells ───────────────────────────────────────────
# In SQL cells, reference widgets with $name syntax
# In Python, build the query string first

dept = dbutils.widgets.get("department")

# Build parameterized query using the widget value
# (in a real notebook this would query an actual table)
query = f"""
SELECT *
FROM   main.default.employees
WHERE  department = '{dept}'
"""
print("Query to run:")
print(query)
# spark.sql(query).show()   # uncomment if the employees table from notebook 02 exists

In [ ]:
# ── Remove widgets ────────────────────────────────────────────────────────────
# Remove a specific widget
dbutils.widgets.remove("regions")

# Remove ALL widgets from the notebook
# dbutils.widgets.removeAll()

print("Widget 'regions' removed.")

---
## Part 7 — dbutils.notebook (Notebook Chaining)

`dbutils.notebook` lets you call one notebook from another — like calling a function.  
This is how Databricks Jobs orchestrate multi-step pipelines using notebooks as tasks.

### Common pattern: orchestrator → worker notebooks

```
main_pipeline  (orchestrator)
    ├─ dbutils.notebook.run("01_ingest",    arguments={"date": "2024-01-01"})
    ├─ dbutils.notebook.run("02_transform", arguments={"env": "prod"})
    └─ dbutils.notebook.run("03_load",      arguments={"table": "sales"})
```

The worker notebook can return a value with `dbutils.notebook.exit("result_string")`.

In [ ]:
# ── dbutils.notebook.run ──────────────────────────────────────────────────────
#
# Syntax:
#   dbutils.notebook.run(
#       path,          # absolute or relative path to the target notebook
#       timeout,       # seconds before this call fails (0 = no timeout)
#       arguments      # dict of widget name → string value
#   )
#
# Returns: whatever string the child notebook passed to dbutils.notebook.exit()
#
# result = dbutils.notebook.run(
#     "/Shared/pipelines/01_ingest",
#     timeout   = 600,
#     arguments = {"run_date": "2024-06-01", "environment": "prod"}
# )
# print(f"Child notebook returned: {result}")

print("dbutils.notebook.run calls a child notebook and waits for it to finish.")
print("The child notebook receives widget values via the 'arguments' dict.")

In [ ]:
# ── dbutils.notebook.exit ─────────────────────────────────────────────────────
# Call this at the END of a worker notebook to return a value to the caller.
# The value must be a STRING (serialize dicts/lists with json.dumps).
# Execution stops immediately when exit() is called.
#
# Example (put this at the bottom of a worker notebook):

import json

result = {
    "status":     "SUCCESS",
    "rows_loaded": 1500,
    "run_date":   "2024-06-01"
}

# In a real worker notebook you would call:
# dbutils.notebook.exit(json.dumps(result))

# In the orchestrator, parse it back:
# raw    = dbutils.notebook.run("worker", 300, {})
# parsed = json.loads(raw)
# print(parsed["rows_loaded"])

print("In a worker notebook, dbutils.notebook.exit() sends a string back to the orchestrator.")
print(f"Example payload: {json.dumps(result)}")

---
## Part 8 — dbutils.jobs (Job Run Context)

When a notebook runs inside a **Databricks Job** (scheduled or triggered), `dbutils.jobs`
lets you retrieve metadata about that run — useful for logging and auditing.

In [ ]:
# ── dbutils.jobs.taskValues ───────────────────────────────────────────────────
# In a multi-task Databricks Job, tasks can share data via taskValues.
# Task A sets a value; Task B reads it.

# Set a value (call this in the upstream task notebook):
# dbutils.jobs.taskValues.set(key="processed_rows", value=1500)

# Get a value set by another task in the same job run:
# rows = dbutils.jobs.taskValues.get(
#     taskKey  = "ingest_task",    # name of the upstream task
#     key      = "processed_rows",
#     default  = 0,                # fallback if key doesn't exist
#     debugValue = 999             # value used when running interactively (not in a job)
# )
# print(f"Rows from upstream task: {rows}")

print("dbutils.jobs.taskValues lets tasks pass values to downstream tasks in the same job.")

---
## Part 9 — DBFS vs Unity Catalog Volumes

| | **DBFS** | **Unity Catalog Volume** |
|---|---|---|
| Path format | `/FileStore/`, `/tmp/`, `/mnt/` | `/Volumes/<catalog>/<schema>/<vol>/` |
| Governance | None — all users share DBFS | Fine-grained ACLs via Unity Catalog |
| Works on UC clusters | `/FileStore/` only; `/tmp/` restricted | Yes — first-class support |
| Backed by | Databricks-managed storage or mounts | Databricks-managed or customer ADLS |
| `dbutils.fs` compatible | Yes | Yes |
| Spark read/write compatible | Yes | Yes |
| Data lineage tracked | No | Yes (in Unity Catalog) |
| Recommended for new work | No | **Yes** |

### Decision rule

```
New workspace with Unity Catalog enabled?
  ├─ Store files      → /Volumes/...           (always)
  ├─ Store tables     → managed or external Delta table in UC
  └─ Legacy code uses /mnt/ or DBFS?  →  leave it; don't migrate unless asked

Old workspace without Unity Catalog?
  ├─ Store files      → /FileStore/ or /mnt/ mount
  └─ Store tables     → Hive metastore
```

---
## Part 10 — Real-world Usage Patterns

These are the patterns you will see and be asked about in interviews.

In [ ]:
# ── Pattern 1: Audit a directory before processing ────────────────────────────
# Check how many files are in a folder, their sizes, and modification times

from datetime import datetime

def audit_directory(path):
    files = dbutils.fs.ls(path)
    total_size = sum(f.size for f in files)
    print(f"Path   : {path}")
    print(f"Files  : {len(files)}")
    print(f"Total  : {total_size:,} bytes")
    for f in files:
        mod_time = datetime.fromtimestamp(f.modificationTime / 1000).strftime("%Y-%m-%d %H:%M")
        print(f"  {f.name:<40} {f.size:>10,} bytes   {mod_time}")

audit_directory(PRACTICE_DIR)

In [ ]:
# ── Pattern 2: Safe write — check before overwriting ─────────────────────────

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

output_path = f"{PRACTICE_DIR}/safe_output/"

if path_exists(output_path):
    print(f"Path already exists: {output_path} — skipping write.")
else:
    df = spark.read.option("header", "true").csv(f"{PRACTICE_DIR}/cities.csv")
    df.write.parquet(output_path)
    print(f"Written to: {output_path}")

In [ ]:
# ── Pattern 3: Parameterized notebook (standard pipeline template) ────────────
# This is the typical pattern for a Databricks Job task notebook

# Step 1 — declare all inputs as widgets at the top
dbutils.widgets.text("pipeline_date", "2024-01-01", "Pipeline Date")
dbutils.widgets.dropdown("layer", "bronze", ["bronze", "silver", "gold"], "Layer")

# Step 2 — read widget values immediately after
pipeline_date = dbutils.widgets.get("pipeline_date")
layer         = dbutils.widgets.get("layer")

print(f"Running pipeline for date={pipeline_date}, layer={layer}")

# Step 3 — use values in your logic
output_path = f"/Volumes/main/default/practice_vol/{layer}/{pipeline_date}/"
print(f"Output will go to: {output_path}")

# Step 4 — at the very end, exit with a status for the orchestrator
# dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "date": pipeline_date}))

---
## Cleanup

In [ ]:
# Remove all widgets created in this notebook
dbutils.widgets.removeAll()

# Remove practice files from FileStore
dbutils.fs.rm(PRACTICE_DIR, recurse=True)
print(f"Cleaned up: {PRACTICE_DIR}")

---
## Key Takeaways

| Utility | Most used command | What it does |
|---|---|---|
| `dbutils.fs.ls` | `dbutils.fs.ls(path)` | List files — returns FileInfo list |
| `dbutils.fs.put` | `dbutils.fs.put(path, text, overwrite=True)` | Write a string to a file |
| `dbutils.fs.head` | `dbutils.fs.head(path, 1024)` | Peek at file contents |
| `dbutils.fs.cp` | `dbutils.fs.cp(src, dst, recurse=True)` | Copy file or directory |
| `dbutils.fs.mv` | `dbutils.fs.mv(src, dst)` | Move or rename |
| `dbutils.fs.rm` | `dbutils.fs.rm(path, recurse=True)` | Delete file or directory |
| `dbutils.secrets.get` | `dbutils.secrets.get("scope", "key")` | Retrieve a secret (never prints) |
| `dbutils.widgets.text` | `dbutils.widgets.text("name", "default", "Label")` | Create a text input |
| `dbutils.widgets.get` | `dbutils.widgets.get("name")` | Read widget value (always string) |
| `dbutils.notebook.run` | `dbutils.notebook.run(path, timeout, args)` | Run a child notebook |
| `dbutils.notebook.exit` | `dbutils.notebook.exit("result")` | Return value to caller |
| `dbutils.jobs.taskValues` | `.set(key, value)` / `.get(taskKey, key)` | Pass data between job tasks |

### Interview one-liners
- **DBFS** = virtual layer over cloud storage; paths like `/mnt/` or `/FileStore/`; mounts are legacy
- **Volumes** = Unity Catalog managed paths (`/Volumes/`); replace mounts; support fine-grained ACLs
- **Secrets** = always use `dbutils.secrets.get`, never hardcode credentials; values are always redacted in output
- **Widgets** = parameterize notebooks for ADF / Databricks Jobs; values always come in as strings
- **Notebook chaining** = `dbutils.notebook.run` calls a child; `dbutils.notebook.exit` returns a value